In [5]:
import subprocess
import re
import numpy as np


def parse_gradient(output, label):
    """Extract the vector from 'label = [ 1.0e+00, 2.0e+00, ... ]' (1D)."""
    m = re.search(rf"{re.escape(label)}\s*=\s*\[(.*?)\]", output, re.DOTALL)
    assert m, f"could not find '{label} = [...]' in output:\n{output}"
    body = m.group(1)
    return np.array([float(tok) for tok in body.split(",") if tok.strip()])

def parse_positions(output, label):
    """Extract '(a, b, c), (d, e, f), ...' from 'label = [ ... ]' into an (M,3) array."""
    m = re.search(rf"{re.escape(label)}\s*=\s*\[(.*?)\]", output, re.DOTALL)
    assert m, f"could not find '{label} = [...]' in output:\n{output}"
    body = m.group(1)
    # grab every (x, y, z) triple
    triples = re.findall(r"\(([^)]*)\)", body)
    rows = [[float(v) for v in t.split(",")] for t in triples]
    return np.array(rows)

def check_match(name, a, b, tol=1e-6):
    """Assert two arrays match within tol; print a one-line verdict."""
    assert a.shape == b.shape, f"{name}: shape mismatch {a.shape} vs {b.shape}"
    max_abs = np.max(np.abs(a - b))
    ok = max_abs < tol
    status = "OK " if ok else "FAIL"
    print(f"  [{status}] {name}: max abs diff = {max_abs:.3e}  (tol {tol:.0e})")
    return ok

def compare_vectors(p1, p2, label1="p1", label2="p2"):
    p1 = np.asarray(p1, dtype=float).ravel()
    p2 = np.asarray(p2, dtype=float).ravel()
    assert p1.shape == p2.shape, f"shape mismatch: {p1.shape} vs {p2.shape}"
    diff = p1 - p2

    print(f"{'i':>3} {label1:>16} {label2:>16} {'diff':>16} "
          f"{'rel_err':>12} {'ratio p2/p1':>14}")
    for i in range(len(p1)):
        d   = diff[i]
        rel = abs(d) / max(abs(p1[i]), abs(p2[i]), 1e-30)
        ratio = p2[i] / p1[i] if abs(p1[i]) > 1e-30 else float("nan")
        print(f"{i:>3} {p1[i]:16.6e} {p2[i]:16.6e} {d:+16.6e} "
              f"{rel:12.4e} {ratio:14.6f}")

    print()
    print(f"max abs diff:     {np.max(np.abs(diff)):.6e}")
    print(f"mean abs diff:    {np.mean(np.abs(diff)):.6e}")
    print(f"max relative err: {np.max(np.abs(diff) / np.maximum(np.abs(p1), 1e-30)):.6e}")
    cos = np.dot(p1, p2) / (np.linalg.norm(p1) * np.linalg.norm(p2))
    print(f"\ncosine similarity:   {cos:.10f}")
    print(f"ratio of norms |{label2}|/|{label1}|: {np.linalg.norm(p2) / np.linalg.norm(p1):.10f}")
    valid = np.abs(p1) > 1e-30
    if np.any(valid):
        ratios = p2[valid] / p1[valid]
        print(f"element-wise ratio: mean={ratios.mean():.6f}, "
              f"std={ratios.std():.6f}, min={ratios.min():.6f}, max={ratios.max():.6f}")

def run(cmd, cwd=None):
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"--- command failed: {cmd} ---")
        print("STDOUT:\n", result.stdout)
        print("STDERR:\n", result.stderr)
        raise RuntimeError(f"{cmd} exited with code {result.returncode}")
    return result.stdout

CPP_EXE    = r"C:\Users\Workstation\borsa_verona\DiffXPBD\build\bin\Release\xpbd.exe"
JAX_SCRIPT = r"C:\Users\Workstation\borsa_verona\DiffXPBD\jax_impl.py"

In [23]:
print("Running C++ ...")
cpp_out = run([CPP_EXE])
print("Running JAX ...")
jax_out = run(["python", JAX_SCRIPT])

# --- forward check: positions must match ---
print("\n" + "=" * 60)
print("FORWARD CHECK (positions should match)")
print("=" * 60)

cpp_final = parse_positions(cpp_out, "pos_final")
jax_final = parse_positions(jax_out, "pos_final")
cpp_guess = parse_positions(cpp_out, "pos_guess")
jax_guess = parse_positions(jax_out, "pos_guess")

POS_TOL = 1e-6
ok_final = check_match("target final positions", cpp_final, jax_final, POS_TOL)
ok_guess = check_match("guess final positions",  cpp_guess, jax_guess, POS_TOL)

if not (ok_final and ok_guess):
    print("\n  WARNING: forward sims diverge — gradient comparison may be meaningless.")

# --- backward check: gradient comparison ---
print("\n" + "=" * 60)
print("GRADIENT COMPARISON")
print("=" * 60)

cpp_grad = parse_gradient(cpp_out, "dL_dalpha")
jax_grad = parse_gradient(jax_out, "dL_dalpha")
compare_vectors(cpp_grad, jax_grad, label1="C++", label2="JAX")

Running C++ ...
Running JAX ...

FORWARD CHECK (positions should match)
  [OK ] target final positions: max abs diff = 0.000e+00  (tol 1e-06)
  [OK ] guess final positions: max abs diff = 0.000e+00  (tol 1e-06)

GRADIENT COMPARISON
  i              C++              JAX             diff      rel_err    ratio p2/p1
  0     7.712715e+01     7.712715e+01    +0.000000e+00   0.0000e+00       1.000000
  1     7.246737e+01     7.246737e+01    +0.000000e+00   0.0000e+00       1.000000
  2     6.909100e+01     6.909100e+01    +0.000000e+00   0.0000e+00       1.000000
  3     7.097999e+01     7.097999e+01    +0.000000e+00   0.0000e+00       1.000000
  4     6.458815e+01     6.458815e+01    +0.000000e+00   0.0000e+00       1.000000
  5     5.228591e+01     5.228591e+01    +0.000000e+00   0.0000e+00       1.000000
  6     2.889561e+01     2.889561e+01    +0.000000e+00   0.0000e+00       1.000000
  7     1.286784e-01     1.286784e-01    +0.000000e+00   0.0000e+00       1.000000
  8     6.751002e+00 

In [18]:
# --- d loss / d x0 comparison ---
print("Running C++ ...")
cpp_out = run([CPP_EXE])
print("Running JAX ...")
jax_out = run(["python", JAX_SCRIPT])

# Optional: also check that the loss matches before trusting the gradient.
def parse_loss(output):
    m = re.search(r"loss:\s*([-\d.eE+]+)", output)
    assert m, f"could not find 'loss: ...' in output"
    return float(m.group(1))

cpp_loss = parse_loss(cpp_out)
jax_loss = parse_loss(jax_out)
print("\n" + "=" * 60)
print("LOSS CHECK")
print("=" * 60)
print(f"  C++ loss = {cpp_loss:.8e}")
print(f"  JAX loss = {jax_loss:.8e}")
print(f"  rel diff = {abs(cpp_loss - jax_loss) / max(abs(jax_loss), 1e-30):.3e}")

# --- gradient ---
print("\n" + "=" * 60)
print("dL/dx0 COMPARISON")
print("=" * 60)

cpp_grad = parse_gradient(cpp_out, "dL_dx0")
jax_grad = parse_gradient(jax_out, "dL_dx0")
compare_vectors(cpp_grad, jax_grad, label1="C++", label2="JAX")

Running C++ ...
Running JAX ...

LOSS CHECK
  C++ loss = 2.51760910e-04
  JAX loss = 2.51760910e-04
  rel diff = 0.000e+00

dL/dx0 COMPARISON
  i              C++              JAX             diff      rel_err    ratio p2/p1
  0     3.346445e-03     3.346445e-03    +0.000000e+00   0.0000e+00       1.000000
  1     3.541466e-02     3.541466e-02    +0.000000e+00   0.0000e+00       1.000000
  2     6.869351e-03     6.869351e-03    +0.000000e+00   0.0000e+00       1.000000
  3    -3.742847e-05    -3.742847e-05    +0.000000e+00   0.0000e+00       1.000000
  4     3.089102e-04     3.089102e-04    +0.000000e+00   0.0000e+00       1.000000
  5     3.208633e-05     3.208633e-05    +0.000000e+00   0.0000e+00       1.000000
  6    -4.637872e-05    -4.637872e-05    +0.000000e+00   0.0000e+00       1.000000
  7    -2.929838e-03    -2.929838e-03    +0.000000e+00   0.0000e+00       1.000000
  8     1.970431e-05     1.970431e-05    +0.000000e+00   0.0000e+00       1.000000
  9    -4.623677e-05    -4.6

In [ ]:
import jax
import jax.numpy as jnp
from jax import jacobian

jax.config.update("jax_enable_x64", True)

def project_distance_alpha(x_i, x_j, w_i, w_j, rest, alpha_tilde):
    delta = x_i - x_j
    dist  = jnp.linalg.norm(delta)
    n     = delta / jnp.where(dist > 1e-12, dist, 1.0)

    C     = dist - rest
    denom = w_i + w_j + alpha_tilde
    dlam  = -C / denom          # lam = 0

    x_i_new = x_i + dlam * w_i * n
    x_j_new = x_j - dlam * w_j * n
    return x_i_new, x_j_new

def dx_dalpha_jax(x_i, x_j, w_i, w_j, rest, alpha_tilde):
    """Returns d(x_i_new)/d(alpha_tilde) and d(x_j_new)/d(alpha_tilde),
    each a 3-vector."""
    def f(a):
        xi, xj = project_distance_alpha(x_i, x_j, w_i, w_j, rest, a)
        return jnp.concatenate([xi, xj])   # shape (6,)
    d = jacobian(f)(alpha_tilde)           # shape (6,) since alpha_tilde scalar
    return d[:3], d[3:]

def run_case(name, x_i, x_j, w_i, w_j, rest, alpha_tilde):
    x_i = jnp.array(x_i, dtype=jnp.float64)
    x_j = jnp.array(x_j, dtype=jnp.float64)

    xi_new, xj_new = project_distance_alpha(x_i, x_j, w_i, w_j, rest, alpha_tilde)
    dxi, dxj = dx_dalpha_jax(x_i, x_j, w_i, w_j, rest, alpha_tilde)

    print(f"\n=== {name} ===")
    print(f"  inputs: x_i={list(x_i)}, x_j={list(x_j)}, "
          f"w_i={w_i}, w_j={w_j}, rest={rest}, alpha_tilde={alpha_tilde}")
    print(f"  x_i_new = ({xi_new[0]:.10f}, {xi_new[1]:.10f}, {xi_new[2]:.10f})")
    print(f"  x_j_new = ({xj_new[0]:.10f}, {xj_new[1]:.10f}, {xj_new[2]:.10f})")
    print(f"  dx_i/dalpha = ({dxi[0]:.10e}, {dxi[1]:.10e}, {dxi[2]:.10e})")
    print(f"  dx_j/dalpha = ({dxj[0]:.10e}, {dxj[1]:.10e}, {dxj[2]:.10e})")

# ============================================================
#  TEST CASES
# ============================================================

# Case 1: simplest — stretched along x, both free, equal mass
run_case("Case 1: axial stretch, equal mass",
         x_i=[0.0, 0.0, 0.0], x_j=[1.5, 0.0, 0.0],
         w_i=1.0, w_j=1.0, rest=1.0, alpha_tilde=0.1)

# Case 2: compressed (C < 0), checks sign
run_case("Case 2: axial compression",
         x_i=[0.0, 0.0, 0.0], x_j=[0.5, 0.0, 0.0],
         w_i=1.0, w_j=1.0, rest=1.0, alpha_tilde=0.1)

# Case 3: pinned p_i (w_i = 0), only p_j moves
run_case("Case 3: pinned p_i",
         x_i=[0.0, 0.0, 0.0], x_j=[1.5, 0.0, 0.0],
         w_i=0.0, w_j=1.0, rest=1.0, alpha_tilde=0.1)

# Case 4: asymmetric masses
run_case("Case 4: asymmetric mass",
         x_i=[0.0, 0.0, 0.0], x_j=[1.5, 0.0, 0.0],
         w_i=2.0, w_j=0.5, rest=1.0, alpha_tilde=0.1)

# Case 5: full 3D direction (not axis-aligned)
run_case("Case 5: diagonal 3D",
         x_i=[0.0, 0.0, 0.0], x_j=[1.0, 1.0, 1.0],
         w_i=1.0, w_j=1.0, rest=1.0, alpha_tilde=0.1)

# Case 6: tiny alpha_tilde (near-rigid, large gradient)
run_case("Case 6: near-rigid (small alpha)",
         x_i=[0.0, 0.0, 0.0], x_j=[1.2, 0.0, 0.0],
         w_i=1.0, w_j=1.0, rest=1.0, alpha_tilde=1e-4)


=== Case 1: axial stretch, equal mass ===
  inputs: x_i=[Array(0., dtype=float64), Array(0., dtype=float64), Array(0., dtype=float64)], x_j=[Array(1.5, dtype=float64), Array(0., dtype=float64), Array(0., dtype=float64)], w_i=1.0, w_j=1.0, rest=1.0, alpha_tilde=0.1
  x_i_new = (0.2380952381, 0.0000000000, 0.0000000000)
  x_j_new = (1.2619047619, 0.0000000000, 0.0000000000)
  dx_i/dalpha = (-1.1337868481e-01, 0.0000000000e+00, 0.0000000000e+00)
  dx_j/dalpha = (1.1337868481e-01, 0.0000000000e+00, 0.0000000000e+00)

=== Case 2: axial compression ===
  inputs: x_i=[Array(0., dtype=float64), Array(0., dtype=float64), Array(0., dtype=float64)], x_j=[Array(0.5, dtype=float64), Array(0., dtype=float64), Array(0., dtype=float64)], w_i=1.0, w_j=1.0, rest=1.0, alpha_tilde=0.1
  x_i_new = (-0.2380952381, 0.0000000000, 0.0000000000)
  x_j_new = (0.7380952381, 0.0000000000, 0.0000000000)
  dx_i/dalpha = (1.1337868481e-01, -0.0000000000e+00, -0.0000000000e+00)
  dx_j/dalpha = (-1.1337868481e-01, -0.